<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Document_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Document Loader

 in this note book i am going to implement pipline for injection and preprocessing of `pdf` using `PyMupdf `

***Script that ingests a PDF, extracts text, splits into question-level chunks, and saves as structured JSON — zero libraries except PyMuPDF.***

 ### setup

In [1]:
%%capture
!pip install pymupdf

In [2]:
import pymupdf
print(pymupdf.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.12 running on linux (64-bit).



In [3]:
%%capture
!pip install -U langchain-text-splitters

In [4]:
%%capture
!pip install langchain-community

## scripts for opening and preprocessing the pdf document

In [6]:
# loads the document
doc = pymupdf.open("/content/Hello transformers.pdf")

Extracting text from Pdf

In [7]:
doc = pymupdf.open("/content/Hello transformers.pdf")
# cretating a text output
out = open("extracted_text.txt","wb")
# iterating over pages
for page in doc:
  tp = page.get_textpage_ocr()
  text = page.get_text(textpage = tp).encode("utf8") # get plain text (is in UTF-8)
  out.write(text) # write text of pages
  out.write(f"\n--- Page {page.number + 1} ---\n".encode("utf8"))

out.close()

In [8]:
with open("extracted_text.txt","r") as f:
    full_document_content = f.read()
# The `extracted_doc` variable (file object) is no longer needed after reading its content into `full_document_content`.

## Chunking the text

In [16]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 512, chunk_overlap = 100)
chunks = text_splitter.split_text(full_document_content)


print(f"Total chunks: {len(chunks)}")
print(chunks[0])



Total chunks: 273
1 A. Vaswani et al., “Attention Is All You Need”, (2017). This title was so catchy that no less than 50 follow-up
papers have included “all you need” in their titles!
2 J. Howard and S. Ruder, “Universal Language Model Fine-Tuning for Text Classification”, (2018).
3 A. Radford et al., “Improving Language Understanding by Generative Pre-Training”, (2018).
4 J. Devlin et al., “BERT: Pre-Training of Deep Bidirectional Transformers for Language Understanding”,
(2018).
CHAPTER 1
Hello Transformers


## Adding Metadata

In [17]:
from datetime import date
from langchain_core.documents import Document

# ── fill these for your document ──────────────────────────────────────
SOURCE_PATH  = "Hello transformer.pdf"
DOC_TYPE     = "book"          # e.g. "report", "invoice", "manual"
DOC_TITLE    = "transformer"
DOC_AUTHOR   = "Harsh Prajapati"
CREATION_DATE = "2005-12-20"       # ISO format from pdf.metadata["creationDate"]
TOTAL_PAGES  = 1                  # from pymupdf doc.page_count

total  = len(chunks)
today  = date.today().isoformat()

# wrap each plain string → Document with full metadata
docs = []
for idx, chunk_text in enumerate(chunks):
    meta = {
        # document-level (same for every chunk)
        "source":         SOURCE_PATH,
        "title":          DOC_TITLE,
        "author":         DOC_AUTHOR,
        "creation_date":  CREATION_DATE,
        "total_pages":    TOTAL_PAGES,
        "doc_type":       DOC_TYPE,

        # chunk-level (unique per chunk)
        "chunk_index":    idx,
        "total_chunks":   total,
        "char_count":     len(chunk_text),
        "prev_chunk_id":  idx - 1 if idx > 0          else None,
        "next_chunk_id":  idx + 1 if idx < total - 1  else None,
        "ingestion_date": today,
    }
    docs.append(Document(page_content=chunk_text, metadata=meta))

# verify
print(f"Total documents: {len(docs)}")
print(f"\nSample metadata (chunk 271):")
for k, v in docs[1].metadata.items():
    print(f"  {k:<18} {v}")
print(f"\nSample content (chunk 271):\n{docs[1].page_content}")

Total documents: 273

Sample metadata (chunk 271):
  source             Hello transformer.pdf
  title              transformer
  author             Harsh Prajapati
  creation_date      2005-12-20
  total_pages        1
  doc_type           book
  chunk_index        1
  total_chunks       273
  char_count         501
  prev_chunk_id      0
  next_chunk_id      2
  ingestion_date     2026-04-13

Sample content (chunk 271):
(2018).
CHAPTER 1
Hello Transformers
In 2017, researchers at Google published a paper that proposed a novel neural net‐
work architecture for sequence modeling.1 Dubbed the Transformer, this architecture
outperformed recurrent neural networks (RNNs) on machine translation tasks, both
in terms of translation quality and training cost.
In parallel, an effective transfer learning method called ULMFiT showed that training
long short-term memory (LSTM) networks on a very large and diverse corpus could


our dataloader is ready !!

print 20 random chunks

In [18]:
for doc in docs[:20]:
    print("----")
    print(doc.page_content)
    print(doc.metadata)

----
1 A. Vaswani et al., “Attention Is All You Need”, (2017). This title was so catchy that no less than 50 follow-up
papers have included “all you need” in their titles!
2 J. Howard and S. Ruder, “Universal Language Model Fine-Tuning for Text Classification”, (2018).
3 A. Radford et al., “Improving Language Understanding by Generative Pre-Training”, (2018).
4 J. Devlin et al., “BERT: Pre-Training of Deep Bidirectional Transformers for Language Understanding”,
(2018).
CHAPTER 1
Hello Transformers
{'source': 'Hello transformer.pdf', 'title': 'transformer', 'author': 'Harsh Prajapati', 'creation_date': '2005-12-20', 'total_pages': 1, 'doc_type': 'book', 'chunk_index': 0, 'total_chunks': 273, 'char_count': 497, 'prev_chunk_id': None, 'next_chunk_id': 1, 'ingestion_date': '2026-04-13'}
----
(2018).
CHAPTER 1
Hello Transformers
In 2017, researchers at Google published a paper that proposed a novel neural net‐
work architecture for sequence modeling.1 Dubbed the Transformer, this architectu

finding text in document using pymupdf


In [21]:
# -*- coding: utf-8 -*-
import pymupdf

# the document to annotate
doc = pymupdf.open("/content/Hello transformers.pdf")

# the text to be marked
needle = """Hello Transformers"""

# work with first page only
page = doc[0]

# get list of text locations
# we use "quads", not rectangles because text may be tilted!
rl = page.search_for(needle, quads=True)

# mark all found quads with one annotation
page.add_squiggly_annot(rl)

# save to a new PDF
doc.save("searched_text.pdf")